In [2]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in this directory:")
print(os.listdir())

Current working directory:
C:\Users\hp\data_science projects\ml zoomcamp

Files in this directory:
['.ipynb_checkpoints', 'car pewdiction.ipynb', 'car_fuel_efficiency_2026.csv', 'data.csv', 'homework 2.ipynb', 'homework1.ipynb', 'homework3.ipynb', 'homework4.ipynb', 'homework5.ipynb']


In [3]:
import os

homework_path = os.path.join(os.getcwd(), "cohorts", "2026", "homework", "05-deployment")

print("Does the deployment folder exist?", os.path.exists(homework_path))

Does the deployment folder exist? False


In [4]:
import os

deployment_path = r"C:\Users\hp\Downloads\machine-learning-zoomcamp-main\machine-learning-zoomcamp-main\cohorts\2026\homework\05-deployment"

print(os.path.exists(deployment_path))

True


In [5]:
import os

for file in sorted(os.listdir(deployment_path)):
    print(file)

.python-version
Dockerfile
README.md
__pycache__
feature_defaults.json
homework.md
homework.yaml
model.py
model_metadata.json
pipeline.bin
predict.py
pyproject.toml
q6_test.py
smoke_test.py
train.py
uv.lock


In [6]:
import shutil

uv_path = shutil.which("uv")

if uv_path:
    print("uv executable found:", uv_path)
else:
    print("uv is not available on the current PATH.")

uv executable found: C:\Users\hp\AppData\Local\Programs\Python\Python313\Scripts\uv.EXE


In [7]:
import os

possible_paths = [
    os.path.expandvars(r"%USERPROFILE%\.local\bin\uv.exe"),
    os.path.expandvars(r"%USERPROFILE%\.cargo\bin\uv.exe"),
    os.path.expandvars(r"%APPDATA%\Python\Scripts\uv.exe"),
]

for path in possible_paths:
    print(path, "->", os.path.exists(path))

C:\Users\hp\.local\bin\uv.exe -> False
C:\Users\hp\.cargo\bin\uv.exe -> False
C:\Users\hp\AppData\Roaming\Python\Scripts\uv.exe -> False


In [8]:
from pathlib import Path

lockfile_path = Path(deployment_path) / "uv.lock"

print("Lockfile exists:", lockfile_path.exists())

if lockfile_path.exists():
    lock_text = lockfile_path.read_text(encoding="utf-8")

    for section in lock_text.split('[[package]]'):
        if 'name = "scikit-learn"' in section:
            for line in section.splitlines():
                if line.strip().startswith("version ="):
                    print("Scikit-learn version:", line.split("=", 1)[1].strip())
                    break
            break
    else:
        print("scikit-learn entry not found.")

Lockfile exists: True
Scikit-learn version: "0.1.0"


In [9]:
import tomllib
from pathlib import Path

lockfile_path = Path(deployment_path) / "uv.lock"

with open(lockfile_path, "rb") as f:
    lock_data = tomllib.load(f)

for package in lock_data["package"]:
    if package["name"] == "scikit-learn":
        print("Scikit-learn version:", package["version"])
        break
else:
    print("Scikit-learn was not found in uv.lock")

Scikit-learn version: 1.7.2


In [10]:
from pathlib import Path

model_file = Path(deployment_path) / "model.py"

print(model_file.read_text())

from __future__ import annotations

import json
from pathlib import Path
from typing import Any


CATEGORICAL_FEATURES = (
    "lead_source",
    "industry",
    "employment_status",
    "location",
)
NUMERICAL_FEATURES = (
    "number_of_courses_viewed",
    "annual_income",
    "interaction_count",
    "lead_score",
)
FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
DEFAULTS_PATH = Path(__file__).with_name("feature_defaults.json")


def load_feature_defaults(path: Path = DEFAULTS_PATH) -> dict[str, float]:
    if not path.exists():
        return {feature: 0.0 for feature in NUMERICAL_FEATURES}
    return json.loads(path.read_text(encoding="utf-8"))


def normalize_record(
    raw: dict[str, Any], numeric_defaults: dict[str, float] | None = None
) -> dict[str, Any]:
    """Apply the training-time missing-value policy to one lead."""

    if numeric_defaults is None:
        numeric_defaults = load_feature_defaults()

    record: dict[str, Any] = {}
    for feature in CATEGORICAL_

In [11]:
import pickle
import sys
from pathlib import Path

# Add the homework folder so Python can import model.py
if str(deployment_path) not in sys.path:
    sys.path.insert(0, str(deployment_path))

from model import normalize_record

# Load the trained pipeline
model_path = Path(deployment_path) / "pipeline.bin"

with open(model_path, "rb") as f:
    pipeline = pickle.load(f)

# First lead record from the homework
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

# Prepare the record and calculate its conversion probability
normalized_lead = normalize_record(lead)
probability = pipeline.predict_proba([normalized_lead])[0, 1]

print("Exact probability:", probability)
print("Rounded probability:", round(probability, 3))


C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Exact probability: 0.5329944186432507
Rounded probability: 0.533


C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [12]:
from pathlib import Path

predict_file = Path(deployment_path) / "predict.py"
print(predict_file.read_text())

from __future__ import annotations

import hashlib
import os
import pickle
from pathlib import Path
from typing import Any

from fastapi import FastAPI
from pydantic import BaseModel

from model import DEFAULTS_PATH, normalize_record


MODEL_PATH = Path(os.getenv("MODEL_PATH", Path(__file__).with_name("pipeline.bin")))
with MODEL_PATH.open("rb") as model_file:
    pipeline = pickle.load(model_file)
MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
FEATURE_DEFAULTS_SHA256 = hashlib.sha256(DEFAULTS_PATH.read_bytes()).hexdigest()

app = FastAPI(title="ML Zoomcamp 2026 lead scoring")


class Lead(BaseModel):
    lead_source: str | None = None
    industry: str | None = None
    employment_status: str | None = None
    location: str | None = None
    number_of_courses_viewed: int | None = None
    annual_income: float | None = None
    interaction_count: int | None = None
    lead_score: float | None = None


@app.get("/health")
def health() -> dict[str, Any]:
    return {


In [13]:

import importlib.util

for package in ["fastapi", "uvicorn", "requests"]:
    print(f"{package}: {'installed' if importlib.util.find_spec(package) else 'not installed'}")


fastapi: not installed
uvicorn: not installed
requests: installed


In [14]:
import sys
import shutil

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("uv executable:", shutil.which("uv"))

Python executable: C:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe
Python version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
uv executable: C:\Users\hp\AppData\Local\Programs\Python\Python313\Scripts\uv.EXE


In [15]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "uv"],
    capture_output=True,
    text=True
)

print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])

In [16]:
import sys
from pathlib import Path

scripts_folder = Path(sys.executable).parent / "Scripts"

print("Scripts folder:", scripts_folder)
print("uv.exe exists:", (scripts_folder / "uv.exe").exists())
print("uv.cmd exists:", (scripts_folder / "uv.cmd").exists())

Scripts folder: C:\Users\hp\AppData\Local\Programs\Python\Python313\Scripts
uv.exe exists: True
uv.cmd exists: False


In [18]:
import subprocess
from pathlib import Path

uv_exe = Path(sys.executable).parent / "Scripts" / "uv.exe"

# Confirm the installed uv version
result = subprocess.run(
    [str(uv_exe), "--version"],
    capture_output=True,
    text=True
)
print(result.stdout)

# Check the Python version required by the homework
result = subprocess.run(
    [str(uv_exe), "python", "find", "3.11"],
    cwd=deployment_path,
    capture_output=True,
    text=True
)
print("Python 3.11 check:")
print(result.stdout)
print(result.stderr)


uv 0.13.6 (329bf9448 2026-10-26 x86_64-pc-windows-msvc)

Python 3.11 check:

error: No interpreter found for Python 3.11 in virtual environments, managed installations, search path, or registry



In [20]:
result = subprocess.run(
    [str(uv_exe), "python", "install", "3.11"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    print("Python installation failed.")
else:
    print("Python installation completed.")


Python 3.11 is already installed

Python installation completed.


In [21]:
import subprocess
from pathlib import Path

result = subprocess.run(
    [str(uv_exe), "python", "find", "3.11"],
    cwd=deployment_path,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Errors:", result.stderr)

Return code: 0
Output: C:\Users\hp\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\python.exe

Errors: 


In [22]:
result = subprocess.run(
    [str(uv_exe), "sync", "--python", "3.11"],
    cwd=deployment_path,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

Return code: 0

Using CPython 3.11.18
Creating virtual environment at: .venv
Resolved 19 packages in 2ms
Prepared 18 packages in 1m 40s
Installed 18 packages in 1.19s
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0



In [23]:
import subprocess
import time
import sys
from pathlib import Path

venv_python = Path(deployment_path) / ".venv" / "Scripts" / "python.exe"

api_process = subprocess.Popen(
    [
        str(venv_python),
        "-m", "uvicorn",
        "predict:app",
        "--host", "127.0.0.1",
        "--port", "9696"
    ],
    cwd=deployment_path,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

print("API process running:", api_process.poll() is None)

API process running: True


In [24]:
import requests

response = requests.get("http://127.0.0.1:9696/health", timeout=10)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [25]:
lead_2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

response = requests.post(
    "http://127.0.0.1:9696/predict",
    json=lead_2,
    timeout=10
)

print("Status code:", response.status_code)
print("Prediction:", response.json())

Status code: 200
Prediction: {'conversion_probability': 0.769799, 'conversion': True}


In [27]:
from pathlib import Path

dockerfile = Path(deployment_path) / "Dockerfile"

for line in dockerfile.read_text().splitlines():
    if line.strip().startswith("FROM "):
        print(line)

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [28]:
import subprocess

result = subprocess.run(
    ["docker", "--version"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [29]:
from pathlib import Path
import os

possible_paths = [
    Path(os.environ.get("ProgramFiles", r"C:\Program Files"))
        / "Docker" / "Docker" / "resources" / "bin" / "docker.exe",
    Path(os.environ.get("LOCALAPPDATA", ""))
        / "Programs" / "Docker" / "Docker" / "resources" / "bin" / "docker.exe",
]

for path in possible_paths:
    print(path, "->", path.exists())

C:\Program Files\Docker\Docker\resources\bin\docker.exe -> False
C:\Users\hp\AppData\Local\Programs\Docker\Docker\resources\bin\docker.exe -> False


In [30]:
import os
from pathlib import Path

search_roots = [
    Path(r"C:\Program Files\Docker"),
    Path(os.environ.get("LOCALAPPDATA", r"C:\Users\hp\AppData\Local")),
]

for root in search_roots:
    print(f"\nSearching: {root}")

    if root.exists():
        matches = list(root.glob("**/docker.exe"))
        for path in matches[:10]:
            print(path)

        if not matches:
            print("No docker.exe found here.")
    else:
        print("Folder does not exist.")


Searching: C:\Program Files\Docker
Folder does not exist.

Searching: C:\Users\hp\AppData\Local
C:\Users\hp\AppData\Local\Programs\DockerDesktop\resources\bin\docker.exe


In [31]:
print(deployment_path)

C:\Users\hp\Downloads\machine-learning-zoomcamp-main\machine-learning-zoomcamp-main\cohorts\2026\homework\05-deployment


In [32]:
import requests

response = requests.get(
    "http://127.0.0.1:9696/health",
    timeout=10
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [33]:
lead_2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

response = requests.post(
    "http://127.0.0.1:9696/predict",
    json=lead_2,
    timeout=10
)

print("Status code:", response.status_code)
print("Prediction:", response.json())

Status code: 200
Prediction: {'conversion_probability': 0.769799, 'conversion': True}
